# Image Denoising Project: kNN vs Averaging Filter

## Problem Statement
Compare image denoising performance between:
1. k-nearest neighbor smoothing (k set by user)
2. Simple averaging filter

Both methods use the same neighborhood window size for fair comparison.
We add uniform noise and evaluate with MSE.

## Setup
Run this once if your environment is missing packages:

`%pip install numpy matplotlib pillow scipy scikit-image scikit-learn ipywidgets`

In [11]:
import io
import time
import numpy as np
import matplotlib.pyplot as plt

from PIL import Image
from scipy.ndimage import uniform_filter
from sklearn.neighbors import NearestNeighbors
from skimage import data as skdata

import ipywidgets as widgets
from IPython.display import display, clear_output

plt.rcParams['figure.figsize'] = (12, 4)
np.random.seed(42)

## Utility Functions

In [12]:
def to_float01(img):
    """Convert uint8 or float image to float32 in [0, 1]."""
    img = np.asarray(img)
    if img.dtype == np.uint8:
        return img.astype(np.float32) / 255.0
    img = img.astype(np.float32)
    if img.max() > 1.0:
        img = img / 255.0
    return np.clip(img, 0.0, 1.0)


def read_uploaded_image(upload_value):
    """Read first image from FileUpload widget value."""
    if not upload_value:
        return None

    if hasattr(upload_value, 'values'):
        first = next(iter(upload_value.values()))
    else:
        first = upload_value[0] if isinstance(upload_value, (list, tuple)) else upload_value

    if isinstance(first, dict):
        content = first['content']
    else:
        content = getattr(first, 'content')

    img = Image.open(io.BytesIO(content))

    if img.mode not in ('L', 'RGB'):
        img = img.convert('RGB')

    return to_float01(np.array(img))


def get_sample_image(name):
    """Load sample images from skimage.data."""
    choices = {
        'camera (grayscale)': skdata.camera(),
        'astronaut (color)': skdata.astronaut(),
        'chelsea (color)': skdata.chelsea(),
        'coffee (color)': skdata.coffee(),
    }
    return to_float01(choices[name])


def add_uniform_noise(img, amplitude):
    """Add uniform noise in [-amplitude, +amplitude] and clip to [0, 1]."""
    noise = np.random.uniform(-amplitude, amplitude, size=img.shape).astype(np.float32)
    return np.clip(img + noise, 0.0, 1.0)


def mse(a, b):
    """Mean squared error between images."""
    return float(np.mean((a.astype(np.float32) - b.astype(np.float32)) ** 2))

## Core Algorithms
- kNN smoothing: for each pixel, look in a local window `w x w`, find `k` nearest intensities to the center intensity, then average them.
- Averaging filter: standard mean filter using the same window size `w`.

In [13]:
def _knn_denoise_channel(channel, k=5, window_size=5):
    """
    Denoise one channel using local-window kNN on intensity values.
    Similarity is intensity distance within the shared local window.
    """
    if window_size % 2 == 0:
        raise ValueError('window_size must be odd.')
    if k < 1:
        raise ValueError('k must be >= 1.')

    h, w = channel.shape
    pad = window_size // 2
    padded = np.pad(channel, pad_width=pad, mode='reflect')
    out = np.empty_like(channel, dtype=np.float32)

    n_candidates = window_size * window_size
    k_eff = min(k, n_candidates)

    for i in range(h):
        for j in range(w):
            patch = padded[i:i + window_size, j:j + window_size].reshape(-1, 1)
            center_val = channel[i, j]

            # Fit local kNN in this patch and query neighbors of center intensity.
            nn = NearestNeighbors(n_neighbors=k_eff, metric='euclidean')
            nn.fit(patch)
            query = np.array([[center_val]], dtype=np.float32)
            idx = nn.kneighbors(query, return_distance=False)[0]
            out[i, j] = np.mean(patch[idx, 0])

    return np.clip(out, 0.0, 1.0)


def knn_denoise_local(image, k=5, window_size=5):
    """Apply local-window kNN denoising to grayscale or RGB image."""
    if image.ndim == 2:
        return _knn_denoise_channel(image, k=k, window_size=window_size)

    channels = [
        _knn_denoise_channel(image[..., c], k=k, window_size=window_size)
        for c in range(image.shape[2])
    ]
    return np.stack(channels, axis=-1)


def averaging_filter(image, window_size=5):
    """Apply mean filter with same window size used for kNN."""
    if window_size % 2 == 0:
        raise ValueError('window_size must be odd.')

    if image.ndim == 2:
        return uniform_filter(image, size=window_size, mode='reflect')

    return np.stack([
        uniform_filter(image[..., c], size=window_size, mode='reflect')
        for c in range(image.shape[2])
    ], axis=-1)

## UI: Input, Parameters, Run, and Visualization

In [ ]:
sample_dropdown = widgets.Dropdown(
    options=['camera (grayscale)', 'astronaut (color)', 'chelsea (color)', 'coffee (color)'],
    value='camera (grayscale)',
    description='Sample:'
)

upload_widget = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Upload Image'
)

noise_slider = widgets.FloatSlider(
    value=0.08, min=0.0, max=0.5, step=0.01,
    description='Noise A:', continuous_update=False
)

k_slider = widgets.IntSlider(
    value=5, min=1, max=49, step=1,
    description='k (kNN):', continuous_update=False
)

window_slider = widgets.IntSlider(
    value=5, min=3, max=15, step=2,
    description='Window w:', continuous_update=False
)

plot_k_curve_checkbox = widgets.Checkbox(
    value=False,
    description='Plot MSE vs k (uses current noisy image)'
)

k_curve_max = widgets.IntSlider(
    value=15, min=3, max=31, step=2,
    description='k max:', continuous_update=False
)

run_button = widgets.Button(description='Run Denoising', button_style='success')
output = widgets.Output()

ui = widgets.VBox([
    widgets.HTML('<h4>Input Section</h4>'),
    sample_dropdown,
    upload_widget,
    widgets.HTML('<h4>Parameter Section</h4>'),
    noise_slider,
    k_slider,
    window_slider,
    plot_k_curve_checkbox,
    k_curve_max,
    widgets.HTML('<h4>Execution Section</h4>'),
    run_button,
    widgets.HTML('<h4>Results Section</h4>'),
    output
])

display(ui)

In [ ]:
def _show_image(ax, img, title):
    if img.ndim == 2:
        ax.imshow(img, cmap='gray', vmin=0, vmax=1)
    else:
        ax.imshow(np.clip(img, 0, 1))
    ax.set_title(title)
    ax.axis('off')


def on_run_clicked(_):
    with output:
        clear_output(wait=True)

        try:
            uploaded = read_uploaded_image(upload_widget.value)
            original = uploaded if uploaded is not None else get_sample_image(sample_dropdown.value)

            noise_amp = float(noise_slider.value)
            k = int(k_slider.value)
            w = int(window_slider.value)

            noisy = add_uniform_noise(original, noise_amp)

            t0 = time.perf_counter()
            knn_img = knn_denoise_local(noisy, k=k, window_size=w)
            knn_time = time.perf_counter() - t0

            t1 = time.perf_counter()
            avg_img = averaging_filter(noisy, window_size=w)
            avg_time = time.perf_counter() - t1

            mse_noisy = mse(original, noisy)
            mse_knn = mse(original, knn_img)
            mse_avg = mse(original, avg_img)

            fig, axes = plt.subplots(1, 4, figsize=(18, 4))
            _show_image(axes[0], original, 'Original')
            _show_image(axes[1], noisy, f'Noisy (A={noise_amp:.2f})')
            _show_image(axes[2], knn_img, f'kNN (k={k}, w={w})\nMSE={mse_knn:.6f}')
            _show_image(axes[3], avg_img, f'Average (w={w})\nMSE={mse_avg:.6f}')
            fig.tight_layout()
            plt.show()

            print('Metrics (lower MSE is better):')
            print(f'- Baseline noisy MSE (original vs noisy): {mse_noisy:.6f}')
            print(f'- kNN denoised MSE (original vs kNN):   {mse_knn:.6f}')
            print(f'- Avg denoised MSE (original vs avg):   {mse_avg:.6f}')
            print('')
            print('Runtime:')
            print(f'- kNN runtime: {knn_time:.3f} s')
            print(f'- Avg runtime: {avg_time:.3f} s')

            if plot_k_curve_checkbox.value:
                max_k = int(k_curve_max.value)
                k_values = [kk for kk in range(1, max_k + 1, 2)]
                mse_curve = []

                t2 = time.perf_counter()
                for kk in k_values:
                    den = knn_denoise_local(noisy, k=kk, window_size=w)
                    mse_curve.append(mse(original, den))
                curve_time = time.perf_counter() - t2

                plt.figure(figsize=(6, 4))
                plt.plot(k_values, mse_curve, marker='o')
                plt.xlabel('k value')
                plt.ylabel('MSE (original vs kNN output)')
                plt.title(f'MSE vs k (window={w})')
                plt.grid(True, alpha=0.3)
                plt.show()

                best_idx = int(np.argmin(mse_curve))
                print(f'Best k on this run: {k_values[best_idx]} with MSE={mse_curve[best_idx]:.6f}')
                print(f'k-curve runtime: {curve_time:.3f} s')

        except Exception as exc:
            print('Error during execution:', exc)


run_button.on_click(on_run_clicked)
print('Ready. Adjust parameters and click Run Denoising.')

Ready. Adjust parameters and click Run Denoising.


## Code Understanding Notes

### What is custom implementation?
- `add_uniform_noise`: custom noise injection in `[-A, +A]`.
- `knn_denoise_local`: custom local-window denoising loop that applies kNN per pixel.
- `mse`: custom metric computation for denoising quality.
- UI callback `on_run_clicked`: custom execution and comparison flow.

### What external libraries do?
- `sklearn.neighbors.NearestNeighbors`: finds nearest intensities in each local window.
- `scipy.ndimage.uniform_filter`: averaging filter baseline.
- `ipywidgets`: interactive controls for input and parameter tuning.
- `matplotlib`: visualization and comparison plots.
- `PIL` and `skimage.data`: image loading and sample datasets.

## Presentation Summary Template
1. Problem statement: remove uniform noise from images.
2. Methodology: compare local-window kNN vs averaging filter with same window size.
3. Implementation details: preprocessing, noise model, denoising pipeline, MSE metric.
4. UI demo: upload image, choose `A`, `k`, `w`, run and compare outputs.
5. Results: report MSE and runtime; include MSE-vs-k curve.
6. Conclusion: discuss quality and speed trade-offs by image type.